In [ ]:
import os
import ee
import geemap
import geopandas as gpd
from pathlib import Path

In [ ]:
# Initialize Earth Engine with your active project
# Explicit Google Cloud Project ID
PROJECT_ID = "weighty-arcadia-488508-k8"

try:
    ee.Initialize(project=PROJECT_ID)
    print(f"Earth Engine successfully initialized with project: {PROJECT_ID}")
except Exception as e:
    print("Initial attempt failed, triggering re-authentication...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine authenticated and initialized successfully.")

In [ ]:
Map = geemap.Map()

In [ ]:

# Load Boundary via Dynamic Relative Path (Native EPSG:4326)
BASE_DIR = Path.cwd()
# If executing from inside the notebooks/ folder, step back to project root
PROJECT_ROOT = BASE_DIR.parent if BASE_DIR.name == "notebooks" else BASE_DIR

bnd_path = os.path.join(
    PROJECT_ROOT, "data", "raw", "ihr_boundary_dissolved.gpkg"
)

# Load layer directly (native EPSG:4326 verified)
gdf_bnd = gpd.read_file(bnd_path)

# Convert GeoPandas geometry directly to Earth Engine FeatureCollection/Geometry
ihr_fc = geemap.gdf_to_ee(gdf_bnd)
ihr_geom = ihr_fc.geometry()

In [ ]:

# Latest available Hansen product covering gross loss events up to 2025
gfc_asset_id = "UMD/hansen/global_forest_change_2025_v1_13"
gfc = ee.Image(gfc_asset_id).clip(ihr_geom)

# Print band details and metadata
band_names = gfc.bandNames().getInfo()
print(f"Hansen GFC Asset Loaded: {gfc_asset_id}")
print(f"Available Bands: {band_names}")

In [ ]:

# 4. Interactive Visualisation via geemap (2001-2025)
Map = geemap.Map()
Map.centerObject(ihr_fc, zoom=6)

# 1. Base Boundary Outline
Map.addLayer(
    ihr_fc, {"color": "black", "fillColor": "00000000", "width": 2}, "IHR Boundary"
)

# 2. Baseline Tree Canopy Cover in Year 2000
treecover = gfc.select("treecover2000")
treecover_vis = {
    "min": 0,
    "max": 100,
    "palette": ["#ffffff", "#a1d99b", "#238b45", "#00441b"],
}
Map.addLayer(treecover, treecover_vis, "Tree Cover 2000 (%)", False)

# 3. Baseline Canopy Mask (>10%) as defined in Indian Forest Survey

baseline_mask = treecover.gte(10).updateMask(gfc.select("datamask").eq(1))
Map.addLayer(
    baseline_mask.selfMask(),
    {"palette": ["#2e7d32"]},
    "Baseline Forest Mask (>10%)",
    False,
)

# 4. Gross Forest Loss (2001-2025)
loss = gfc.select("loss")
loss_vis = {"palette": ["#d73027"]}
Map.addLayer(
    loss.selfMask(), loss_vis, "Gross Forest Loss (2001-2025)", True
)
# 5. Spatiotemporal Loss Progression (Years 1 to 25)
lossyear = gfc.select("lossyear")
lossyear_vis = {
    "min": 1,
    "max": 25,
    "palette": [
        "#ffffcc",  # Early (2001-2005) --> Light Yellow to Deep Green
        "#c2e699",
        "#78c679",
        "#31a354",
        "#006837",
        "#fed976",  # Mid (2006-2015) --> Light Orange to Dark Red
        "#fd8d3c",
        "#f03b20",
        "#bd0026",
        "#7a0177",  # Recent (2016-2025) --> Deep magenta/purple to Eggplant colour
        "#49006a",
    ],
}
Map.addLayer(
    lossyear.selfMask(),
    lossyear_vis,
    "Loss Year (1=2001 to 25=2025)",
    False,
)
Map

### Defining the Spatial Unit
Creating a regular spatial analysis grid covering the IHR. Initially test 2.5 km, 5 km and 10 km candidate scales. 
Harris et al. used 2.5-km bins after testing multiple scales, whereas Singh & Yan used a 10-km neighbourhood.

In [ ]:
import numpy as np
from shapely.geometry import box
import pyproj

### No need to run this more than once, if all the geopackages are already present in the data/raw folder

In [ ]:
def make_grid(gdf_boundary, cell_size_km, id_prefix="C"):
    """
    Create a regular metric grid in a custom IHR-centred LAEA projection
    and retain both the projected metric grid and WGS84 version.
    """

    # 1. Convert boundary to WGS84
    gdf_wgs84 = gdf_boundary.to_crs(4326)

    # 2. Create custom LAEA projection centred on IHR
    centroid = gdf_wgs84.union_all().centroid

    laea_crs = pyproj.CRS.from_proj4(
        f"+proj=laea +lat_0={centroid.y} +lon_0={centroid.x} "
        f"+datum=WGS84 +units=m +no_defs"
    )

    # 3. Project boundary to metric CRS
    gdf_proj = gdf_wgs84.to_crs(laea_crs)
    boundary_union = gdf_proj.union_all()

    # 4. Create regular grid in metres
    minx, miny, maxx, maxy = boundary_union.bounds

    cell_size_m = cell_size_km * 1000

    xs = np.arange(minx, maxx + cell_size_m, cell_size_m)
    ys = np.arange(miny, maxy + cell_size_m, cell_size_m)

    cells = [
        box(x, y, x + cell_size_m, y + cell_size_m)
        for x in xs[:-1]
        for y in ys[:-1]
    ]

    # 5. Create projected/metric grid
    grid_metric = gpd.GeoDataFrame(
        {"geometry": cells},
        crs=laea_crs
    )

    # Keep cells intersecting the IHR
    grid_metric = (
        grid_metric[grid_metric.intersects(boundary_union)]
        .reset_index(drop=True)
    )

    # 6. Add IDs
    grid_metric["Cell_ID"] = [
        f"{id_prefix}{i:06d}"
        for i in range(len(grid_metric))
    ]

    grid_metric["cell_size_km"] = cell_size_km

    # Reorder columns
    grid_metric = grid_metric[
        ["Cell_ID", "cell_size_km", "geometry"]
    ]

    # 7. Create WGS84 copy for GEE/interoperability
    grid_wgs84 = grid_metric.to_crs(4326)

    return grid_metric, grid_wgs84

In [ ]:
grid_2_5km_metric, grid_2_5km = make_grid(
    gdf_bnd,
    2.5,
    id_prefix="G25_"
)

grid_5km_metric, grid_5km = make_grid(
    gdf_bnd,
    5.0,
    id_prefix="G5_"
)

grid_10km_metric, grid_10km = make_grid(
    gdf_bnd,
    10.0,
    id_prefix="G10_"
)

In [ ]:
print("Metric CRS:")
print(grid_2_5km_metric.crs)

print("\nWGS84 CRS:")
print(grid_2_5km.crs)

print("\nCell counts:")
print(len(grid_2_5km_metric), len(grid_2_5km))

#### Avoid the immediate code cell below, if you already have these geopackages saved.

In [ ]:
# Persist candidate grids for reproducibility and later sensitivity testing (Phase J)

grid_out_dir = os.path.join(PROJECT_ROOT, "data", "raw")

# --- 2.5 km grids ---
grid_2_5km_metric.to_file(
    os.path.join(grid_out_dir, "ihr_grid_2_5km_metric.gpkg"),
    driver="GPKG"
)

grid_2_5km.to_file(
    os.path.join(grid_out_dir, "ihr_grid_2_5km_wgs84.gpkg"),
    driver="GPKG"
)

# --- 5 km grids ---
grid_5km_metric.to_file(
    os.path.join(grid_out_dir, "ihr_grid_5km_metric.gpkg"),
    driver="GPKG"
)

grid_5km.to_file(
    os.path.join(grid_out_dir, "ihr_grid_5km_wgs84.gpkg"),
    driver="GPKG"
)

# --- 10 km grids ---
grid_10km_metric.to_file(
    os.path.join(grid_out_dir, "ihr_grid_10km_metric.gpkg"),
    driver="GPKG"
)

grid_10km.to_file(
    os.path.join(grid_out_dir, "ihr_grid_10km_wgs84.gpkg"),
    driver="GPKG"
)

In [ ]:
# Adopt the 2.5-km grid as the primary analysis scale
# grid_primary = grid_2_5km_metric.copy()
grid_primary = grid_2_5km.copy()

# Convert to an Earth Engine FeatureCollection for zonal computation
grid_fc = geemap.gdf_to_ee(grid_primary)

print(f"Primary grid: {len(grid_primary):,} cells")
print("Earth Engine FeatureCollection created successfully.")

#### Local GeoPandas Visualisation

In [ ]:
# Visualize the 2.5-km grid locally
ax = gdf_bnd.boundary.plot(figsize=(12, 12), linewidth=1)
grid_primary.boundary.plot(ax=ax, linewidth=0.1)

ax.set_title("IHR Analysis Grid - 2.5 km")
ax.set_axis_off()

#### Earth Engine Visualisation
We are using 100th cell for visualisation, due to computation limit exhaustion in GEE

In [ ]:
grid_preview = grid_primary.iloc[::100].copy()

grid_preview_fc = geemap.gdf_to_ee(grid_preview)

Map.addLayer(
    ee.Image().paint(grid_preview_fc, 0, 1),
    {"palette": ["#3182bd"]},
    "2.5 km Grid Preview",
    True,
)
Map.centerObject(ihr_fc, zoom=6)

Map